# Dependencies and imports

In [83]:
!pip install google-cloud-bigquery -q
!pip install google-cloud-bigquery google-cloud-aiplatform pandas -q
from google.cloud import bigquery, aiplatform
from vertexai.language_models import TextEmbeddingModel
from google import genai
from google.colab import userdata
import os


In [95]:
from google.cloud import bigquery
import pandas as pd
from google.cloud import aiplatform
from vertexai.language_models import TextEmbeddingModel


# If running in Colab/Vertex, this handles auth automatically
# Otherwise, set your credentials:
# import os
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "path/to/service-account.json"
PROJECT_ID="qwiklabs-gcp-01-2fecb47c3bc2"
DATASET = "AB_Dataset"
VERTEX_LOCATION = "us-east1"
BQ_LOCATION = "US"
TABLE = "aurora_bay_faqs_embedded"
GEMINI_MODEL = "gemini-2.5-flash"
EMBEDDING_MODEL = "embedding_model"

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=VERTEX_LOCATION)
embed_model = TextEmbeddingModel.from_pretrained("text-embedding-005")
location = "us-east1"

print("Ready!")

Ready!


# Import data source

In [8]:

dataset = bigquery.Dataset(f"qwiklabs-gcp-01-2fecb47c3bc2.AB_Dataset")
dataset.location = "US"
client.create_dataset(dataset, exists_ok=True)
print("Dataset ready")
table_id = "qwiklabs-gcp-01-2fecb47c3bc2.AB_Dataset.aurora_bay_faqs"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,       # skip header row
    autodetect=True,           # auto-detect schema from CSV
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE  # overwrite if exists
)

uri = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

load_job = client.load_table_from_uri(uri, table_id, job_config=job_config)
load_job.result()  # waits for job to complete

table = client.get_table(table_id)
print(f"Loaded {table.num_rows} rows into {table_id}")

Dataset ready
Loaded 50 rows into qwiklabs-gcp-01-2fecb47c3bc2.AB_Dataset.aurora_bay_faqs


# Load dataset

In [17]:
DATASET = dataset
TABLE = "aurora_bay_faqs"

df = client.query(f"SELECT * FROM `qwiklabs-gcp-01-2fecb47c3bc2.AB_Dataset.aurora_bay_faqs`").to_dataframe()
df.columns = ["question", "answer"]
print(df.head())


                                         question  \
0                    When was Aurora Bay founded?   
1           What is the population of Aurora Bay?   
2      Where is the Aurora Bay Town Hall located?   
3         Who is the current mayor of Aurora Bay?   
4  What are the primary industries in Aurora Bay?   

                                              answer  
0  Aurora Bay was founded in 1901 by a group of f...  
1  Aurora Bay has a population of approximately 3...  
2  The Town Hall is located at 100 Harbor View Ro...  
3  The current mayor is Linda Greenwood, elected ...  
4  The primary industries include commercial fish...  


# Generate embeddings

In [19]:
aiplatform.init(project=PROJECT_ID, location="us-central1")

model = TextEmbeddingModel.from_pretrained("text-embedding-005")

def get_embedding(text):
    result = model.get_embeddings([text])
    return result[0].values

# Combine question + answer for richer embeddings
df["embedding"] = df.apply(
    lambda r: get_embedding(f"Question: {r['question']} Answer: {r['answer']}"),
    axis=1
)

print(f"Embedding dimension: {len(df['embedding'][0])}")
print(df.head())

/usr/local/lib/python3.12/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Embedding dimension: 768
                                         question  \
0                    When was Aurora Bay founded?   
1           What is the population of Aurora Bay?   
2      Where is the Aurora Bay Town Hall located?   
3         Who is the current mayor of Aurora Bay?   
4  What are the primary industries in Aurora Bay?   

                                              answer  \
0  Aurora Bay was founded in 1901 by a group of f...   
1  Aurora Bay has a population of approximately 3...   
2  The Town Hall is located at 100 Harbor View Ro...   
3  The current mayor is Linda Greenwood, elected ...   
4  The primary industries include commercial fish...   

                                           embedding  
0  [-0.079507015645504, 0.011648421175777912, 0.0...  
1  [-0.032301537692546844, 0.015699131414294243, ...  
2  [-0.06871124356985092, -0.024075346067547798, ...  
3  [-0.0759090706706047, -0.024423591792583466, -...  
4  [-0.03486865386366844, 0.0240632109344005

# Write back to big query

In [21]:
dest_table_id = f"qwiklabs-gcp-01-2fecb47c3bc2.AB_Dataset.aurora_bay_faqs_embedded"

schema = [
    bigquery.SchemaField("question", "STRING"),
    bigquery.SchemaField("answer", "STRING"),
    bigquery.SchemaField("embedding", "FLOAT64", mode="REPEATED"),  # vector stored as array
]

job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

load_job = client.load_table_from_dataframe(df, dest_table_id, job_config=job_config)
load_job.result()

table = client.get_table(dest_table_id)
print(f"Loaded {table.num_rows} rows with embeddings into {dest_table_id}")

Loaded 50 rows with embeddings into qwiklabs-gcp-01-2fecb47c3bc2.AB_Dataset.aurora_bay_faqs_embedded


In [27]:
sample = client.query(f"""
    SELECT question, answer,
           ARRAY_LENGTH(embedding) as embedding_dim
    FROM `{dest_table_id}`
    LIMIT 5
""").to_dataframe()

print(sample)

                                         question  \
0                    When was Aurora Bay founded?   
1           What is the population of Aurora Bay?   
2      Where is the Aurora Bay Town Hall located?   
3         Who is the current mayor of Aurora Bay?   
4  What are the primary industries in Aurora Bay?   

                                              answer  embedding_dim  
0  Aurora Bay was founded in 1901 by a group of f...            768  
1  Aurora Bay has a population of approximately 3...            768  
2  The Town Hall is located at 100 Harbor View Ro...            768  
3  The current mayor is Linda Greenwood, elected ...            768  
4  The primary industries include commercial fish...            768  


# Chatbot

In [88]:
DATASET = "AB_Dataset"
TABLE = "aurora_bay_faqs_embedded"

vertexai.init(project=PROJECT_ID, location="us-central1")
bq_client = bigquery.Client(project=PROJECT_ID)
embed_model = TextEmbeddingModel.from_pretrained("text-embedding-005")

def find_relevant_faqs(user_question, top_k=3):
    # Embed the user's question
    question_embedding = embed_model.get_embeddings([user_question])[0].values

    # BigQuery vector search using ML.DISTANCE
    query = f"""
      SELECT base.content AS content, distance
      FROM VECTOR_SEARCH(
          TABLE `{PROJECT_ID}.{DATASET}.faqs_embedded`,
          'ml_generate_embedding_result',
          (
              SELECT ml_generate_embedding_result, content
              FROM ML.GENERATE_EMBEDDING(
                  MODEL `{PROJECT_ID}.{DATASET}.{EMBEDDING_MODEL}`,
                  (SELECT @q AS content)
              )
          ),
          top_k => {top_k}
      )
      ORDER BY distance
    """

    results = bq_client.query(query).to_dataframe()
    return results

In [89]:


def ask_chatbot(user_question, top_k=3, distance_threshold=0.5):
    # 1. Find relevant FAQs via vector search
    relevant = find_relevant_faqs(user_question, top_k=top_k)

    # 2. Filter out poor matches
    relevant = relevant[relevant["distance"] < distance_threshold]

    if relevant.empty:
        return "I'm sorry, I don't have information about that topic."

    # 3. Build context from retrieved FAQs
    context = "\n\n".join([
        f"Q: {row['question']}\nA: {row['answer']}"
        for _, row in relevant.iterrows()
    ])

    # 4. Send to Gemini with context
    prompt = f"""
    Context from Aurora Bay knowledge base:
    {context}

    User question: {user_question}
    """

    response = gemini.generate_content(prompt)  # same call, different SDK
    return response.text

In [94]:
user_question = "When was Aurora Bay established?"

relevant = find_relevant_faqs(user_question, top_k=3)
relevant = relevant[relevant["distance"] < 0.5]

context = "\n\n".join([
    f"Q: {row['question']}\nA: {row['answer']}"
    for _, row in relevant.iterrows()
])

prompt = f"""
Context from Aurora Bay knowledge base:
{context}

User question: {user_question}
"""

response = gemini.generate_content(prompt)
print(response.text)

NotFound: 404 Not found: Table qwiklabs-gcp-01-2fecb47c3bc2:AB_Dataset.faqs_embedded was not found in location US; reason: notFound, message: Not found: Table qwiklabs-gcp-01-2fecb47c3bc2:AB_Dataset.faqs_embedded was not found in location US

Location: US
Job ID: a0fc95a9-f02c-4faf-a0f9-0c9f85817275


In [71]:
!gcloud services enable generativelanguage.googleapis.com

Operation "operations/acat.p2-1088474518832-3bba3e26-b158-4675-abd7-07a9b8164f5e" finished successfully.
